# SUBCPMK 4 — Preprocessing dan Indexing

Mengindeks seluruh **teks yang tersedia** pada PDF input. Untuk memenuhi ketentuan dosen 'seluruh bagian putusan', wajib mengganti sumber dengan putusan resmi lengkap; 50 PDF ringkasan satu halaman saat ini **tidak cukup** untuk klaim itu.


## 1. Siapkan paket dan akses data
Folder input default `/content/PN_Tanjung_Redeb/{pdf,txt}` di Colab atau folder kerja bersama SUBCPMK 3 di lokal. ZIP standar terdeteksi otomatis; untuk lokasi lain isi `ZIP_MANUAL`. Jalankan SUBCPMK 3 lebih dahulu agar TXT dan audit tersedia.

In [1]:
%pip -q install pymupdf pandas Sastrawi matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
from collections import Counter, defaultdict
from itertools import chain
import math, re, csv, json, os, zipfile, shutil, unicodedata
import pandas as pd
import fitz
try:
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    SASTRAWI_TERSEDIA = True
except ImportError:
    SASTRAWI_TERSEDIA = False
    print('Sastrawi belum terpasang; pengolahan tetap berjalan dengan stopword dasar dan tanpa stemming. Install Sastrawi untuk hasil final.')

def cari_root_proyek(start=None):
    start = (start or Path.cwd()).resolve()
    for folder in (start, *start.parents):
        if (folder / 'README_MULAI_DI_SINI.md').is_file():
            return folder
    return None

COLAB = Path('/content').is_dir()
PROJECT_ROOT = None if COLAB else cari_root_proyek()
ROOT = Path('/content/PN_Tanjung_Redeb') if COLAB else (PROJECT_ROOT / '03_SUBCPMK3_Crawler_dan_Ekstraksi' / 'PN_Tanjung_Redeb' if PROJECT_ROOT else Path.cwd() / 'PN_Tanjung_Redeb')
PDF_DIR, TXT_DIR, OUT_DIR=ROOT/'pdf',ROOT/'txt',ROOT/'hasil_subcpmk4'
for d in (PDF_DIR,TXT_DIR,OUT_DIR):d.mkdir(parents=True,exist_ok=True)
ZIP_MANUAL = ''  # opsional; SUBCPMK 4 lebih dahulu memakai PDF/TXT hasil SUBCPMK 3
zip_candidates = [Path(ZIP_MANUAL)] if ZIP_MANUAL else []
if COLAB:
    zip_candidates += [Path('/content/INPUT_DATASET_50_PDF_UNTUK_COLAB.zip'), Path('/content/50_Putusan_PN_Tanjung_Redeb_Narkotika_Tanpa_BHT.zip'), Path('/content/Narkotika.zip')]
elif PROJECT_ROOT:
    zip_candidates += [PROJECT_ROOT / 'INPUT_DATASET_50_PDF_UNTUK_COLAB.zip', PROJECT_ROOT / '02_SUBCPMK2_Dataset_GitHub' / 'Dataset-Narkotika_XXX_YYY' / 'Dataset' / 'Narkotika.zip']
ZIP_INPUT = next((p.resolve() for p in zip_candidates if p.is_file() and zipfile.is_zipfile(p)), None)
if ZIP_INPUT is not None:
    src=Path(ZIP_INPUT)
    if src.is_file() and zipfile.is_zipfile(src):
        with zipfile.ZipFile(src) as z:
            for info in z.infolist():
                if info.is_dir() or not info.filename.lower().endswith(('.pdf','.txt')):continue
                target_folder=PDF_DIR if info.filename.lower().endswith('.pdf') else TXT_DIR
                target=target_folder/Path(info.filename).name
                if not target.name:continue
                with z.open(info) as fin,open(target,'wb') as fout:shutil.copyfileobj(fin,fout)
else:
    print('ZIP tidak ditemukan; memakai PDF/TXT yang sudah ada dari SUBCPMK 3.')
print('Root proyek:', PROJECT_ROOT or '(Colab/tidak terdeteksi)')
print('Folder data bersama:',ROOT)
print('ZIP terpilih:', ZIP_INPUT or 'tidak ada')

Root proyek: C:\Users\marce\KULIAH\Smt7\InformationRetrival\UTS
Folder data bersama: C:\Users\marce\KULIAH\Smt7\InformationRetrival\UTS\03_SUBCPMK3_Crawler_dan_Ekstraksi\PN_Tanjung_Redeb
ZIP terpilih: C:\Users\marce\KULIAH\Smt7\InformationRetrival\UTS\INPUT_DATASET_50_PDF_UNTUK_COLAB.zip


c:\Users\marce\AppData\Local\Programs\Python\Python312\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


## 2. Baca teks penuh dan metadata
TXT dari SUBCPMK 3 diutamakan; jika tidak tersedia, PDF asli dibaca. Satu dokumen = satu unit pengindeksan. Nama berkas menjadi ID stabil.

In [3]:
def baca_pdf(path):
    try:
        with fitz.open(path) as pdf:
            return '\n'.join(page.get_text('text',sort=True) for page in pdf)
    except Exception as err:
        print('Lewati PDF bermasalah:',path.name,str(err)[:80]);return ''

CRAWLER_PDF_DIR, CRAWLER_TXT_DIR = ROOT/'pdf_crawler', ROOT/'txt_crawler'
ACTIVE_TXT_DIR = CRAWLER_TXT_DIR if any(CRAWLER_TXT_DIR.glob('*.txt')) else TXT_DIR
ACTIVE_PDF_DIR = CRAWLER_PDF_DIR if any(CRAWLER_PDF_DIR.glob('*.pdf')) else PDF_DIR
print('Sumber indeks:', ACTIVE_TXT_DIR, '| fallback PDF:', ACTIVE_PDF_DIR)
DOCS={}
for txt in sorted(ACTIVE_TXT_DIR.glob('*.txt')):
    body=txt.read_text(encoding='utf-8',errors='replace')
    if body.strip():DOCS[txt.stem]=body
for pdf in sorted(ACTIVE_PDF_DIR.glob('*.pdf')):
    if pdf.stem not in DOCS:
        body=baca_pdf(pdf)
        if body.strip():DOCS[pdf.stem]=body
print('Dokumen yang berhasil dibaca:',len(DOCS))
if not DOCS:print('Belum ada PDF/TXT asli. Indeks akan kosong sampai data dimasukkan.')

Dokumen yang berhasil dibaca: 50


## 3. Metadata, statistik, dan preprocessing berbahasa Indonesia
Normalisasi Unicode, case folding, cleaning karakter, tokenisasi, penghapusan stopword, dan stemming Sastrawi. Nomor pasal/perkara dapat tetap diindeks sebagai token angka; nama dokumen tetap sebagai metadata.

In [4]:
NO_RE=re.compile(r'\b(\d{1,5})\s*/\s*Pid\s*\.\s*Sus\s*/\s*(20\d{2})\s*/\s*PN\s*\.?\s*Tnr\b',re.I)
metadata=[]
for file,text in DOCS.items():
    match=NO_RE.search(text[:40000])
    metadata.append({'doc_id':file,'no_putusan':f'{match.group(1)}/Pid.Sus/{match.group(2)}/PN Tnr' if match else '',
                     'lembaga':'PN Tanjung Redeb','panjang_karakter':len(text),
                     'jumlah_kata_raw':len(text.split())})
METADATA=pd.DataFrame(metadata,columns=['doc_id','no_putusan','lembaga','panjang_karakter','jumlah_kata_raw'])
METADATA.to_csv(OUT_DIR/'metadata.csv',index=False,encoding='utf-8-sig')
if not METADATA.empty:display(METADATA.head())
print('CATATAN: indeks hanya memuat teks pada input. PDF ringkasan satu halaman bukan pengganti full text putusan resmi.')

STOPWORDS=(set(StopWordRemoverFactory().get_stop_words()) if SASTRAWI_TERSEDIA else set('yang dan di ke dari pada untuk dengan atau ini itu ada telah adalah dalam sebagai bagi oleh para nya'.split()))
# Kata yang penting dalam konteks hukum jangan dihapus meski terdaftar sebagai stopword
STOPWORDS -= {'putusan','pengadilan','pidana','pasal','terdakwa','hakim','dakwaan'}
STEMMER=StemmerFactory().create_stemmer() if SASTRAWI_TERSEDIA else None
NORMAL={'putusanputusan':'putusan','pengadilannegeri':'pengadilan negeri'}
TOKEN=re.compile(r'(?u)\b[\w]+\b')

def normalisasi(teks):
    teks=unicodedata.normalize('NFKC',teks).lower()
    teks=re.sub(r'https?://\S+',' ',teks)
    teks=re.sub(r'\s+',' ',teks)
    return teks

def tokens(teks,stopword=True,stemming=True):
    kata=TOKEN.findall(normalisasi(teks))
    kata=[NORMAL.get(k,k) for k in kata if len(k)>1 or k.isdigit()]
    if stopword:kata=[k for k in kata if k not in STOPWORDS]
    if stemming:kata=[STEMMER.stem(k) if (STEMMER is not None and k.isalpha()) else k for k in kata]
    return kata

TOKENS={doc:tokens(content) for doc,content in DOCS.items()}
STATS=pd.DataFrame([{'doc_id':doc,'token_count':len(ts),'unique_terms':len(set(ts))}
                    for doc,ts in TOKENS.items()],columns=['doc_id','token_count','unique_terms'])
STATS.to_csv(OUT_DIR/'statistik_dokumen.csv',index=False)
if not STATS.empty:display(STATS.describe(include='all').T)

,doc_id,no_putusan,lembaga,panjang_karakter,jumlah_kata_raw
0,Putusan_102_Pid.Sus_2025_PN Tnr,102/Pid.Sus/2025/PN Tnr,PN Tanjung Redeb,4697,252
1,Putusan_105_Pid.Sus_2023_PN Tnr,105/Pid.Sus/2023/PN Tnr,PN Tanjung Redeb,4622,252
2,Putusan_108_Pid.Sus_2024_PN Tnr,108/Pid.Sus/2024/PN Tnr,PN Tanjung Redeb,4714,250
3,Putusan_111_Pid.Sus_2025_PN Tnr,111/Pid.Sus/2025/PN Tnr,PN Tanjung Redeb,4705,252
4,Putusan_114_Pid.Sus_2023_PN Tnr,114/Pid.Sus/2023/PN Tnr,PN Tanjung Redeb,4717,254


CATATAN: indeks hanya memuat teks pada input. PDF ringkasan satu halaman bukan pengganti full text putusan resmi.


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
doc_id,50,50,Putusan_102_Pid.Sus_2025_PN Tnr,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
token_count,50.0,NaN,NaN,NaN,227.66,1.598596,225.0,227.0,228.0,229.0,230.0
unique_terms,50.0,NaN,NaN,NaN,115.1,1.417817,112.0,114.0,115.0,116.0,119.0


## 4. Tahapan preprocessing pada dokumen nyata
Membandingkan sebelum dan sesudah pada dokumen pertama yang tersedia. Jika kosong, output demo juga kosong, bukan menggunakan teks fiktif.

In [5]:
if DOCS:
    doc_id=next(iter(DOCS))
    text=DOCS[doc_id]
    sample=pd.DataFrame([
        {'tahap':'Original','cuplikan':text[:200],'banyak_token':len(text.split())},
        {'tahap':'Normalisasi','cuplikan':normalisasi(text)[:200],'banyak_token':len(tokens(text,False,False))},
        {'tahap':'Stopword removal','cuplikan':' '.join(tokens(text,True,False)[:35]),'banyak_token':len(tokens(text,True,False))},
        {'tahap':'Stemming','cuplikan':' '.join(TOKENS[doc_id][:35]),'banyak_token':len(TOKENS[doc_id])},
    ])
    display(sample)
    sample.to_csv(OUT_DIR/'contoh_preprocessing.csv',index=False,encoding='utf-8-sig')
else:
    print('Belum ada contoh dokumen nyata untuk dibandingkan.')

,tahap,cuplikan,banyak_token
0,Original,PUTUSAN PENGADILAN NEGERI TANJUNG REDE...,252
1,Normalisasi,putusan pengadilan negeri tanjung redeb nomor...,254
2,Stopword removal,putusan pengadilan negeri tanjung redeb nomor ...,226
3,Stemming,putus adil negeri tanjung redeb nomor 102 pid ...,226


## 5. Bangun positional inverted index + bobot BM25 dan TF-IDF
`term → doc_id → [posisi term]`. Posisi memungkinkan pemeriksaan term yang cocok. Ranker diimplementasikan independen dengan rumus BM25 dan cosine TF-IDF.

In [6]:
POSTINGS=defaultdict(dict)
DOC_LENGTH={doc:len(toks) for doc,toks in TOKENS.items()}
N=len(TOKENS)
AVGDL=(sum(DOC_LENGTH.values())/N) if N else 0.0
for doc_id,toks in TOKENS.items():
    local=defaultdict(list)
    for pos,term in enumerate(toks):local[term].append(pos)
    for term,poslist in local.items():POSTINGS[term][doc_id]=poslist
DF={term:len(p) for term,p in POSTINGS.items()}
IDF={term:math.log((N-df+0.5)/(df+0.5)+1) for term,df in DF.items()}
print('N:',N,'| Vocabulary:',len(DF),'| Rata-rata token:',round(AVGDL,1))

# Simpan indeks lengkap (ukuran bisa besar untuk 50 putusan panjang)
with open(OUT_DIR/'inverted_index.json','w',encoding='utf-8') as f:
    json.dump({'N':N,'avgdl':AVGDL,'postings':POSTINGS,'df':DF},f,ensure_ascii=False)
pd.DataFrame([{'term':term,'document_frequency':df,'idf_bm25':IDF[term]}
              for term,df in sorted(DF.items())],columns=['term','document_frequency','idf_bm25']).to_csv(
                  OUT_DIR/'vocabulary.csv',index=False)

N: 50 | Vocabulary: 361 | Rata-rata token: 227.7


## 6. Uji pencarian teks lengkap
Ubah `QUERY` sesuai kebutuhan. Skor 0 bukan dokumen relevan. Hasil selalu merujuk ke `doc_id` putusan asli.

In [7]:
def cari(query,metode='bm25',top_k=5):
    q=Counter(tokens(query))
    if not q or not N:return pd.DataFrame(columns=['rank','doc_id','no_putusan','score'])
    nilai=defaultdict(float)
    k1,b=1.5,0.75
    if metode == 'bm25':
        for term,qf in q.items():
            for doc,positions in POSTINGS.get(term,{}).items():
                tf=len(positions); denom=tf+k1*(1-b+b*DOC_LENGTH[doc]/(AVGDL or 1))
                nilai[doc]+=IDF[term]*(tf*(k1+1)/denom)*qf
    elif metode=='tfidf':
        idf={t:math.log((1+N)/(1+DF[t]))+1 for t in DF}
        query_weights={t:(1+math.log(f))*idf.get(t,0) for t,f in q.items()}
        doc_norm={}
        for doc,ts in TOKENS.items():
            freq=Counter(ts)
            doc_norm[doc]=math.sqrt(sum(((1+math.log(f))*idf[t])**2 for t,f in freq.items())) or 1
        qnorm=math.sqrt(sum(x*x for x in query_weights.values())) or 1
        for term,qweight in query_weights.items():
            for doc,positions in POSTINGS.get(term,{}).items():
                doc_weight=(1+math.log(len(positions)))*idf[term]
                nilai[doc]+=doc_weight*qweight/(doc_norm[doc]*qnorm)
    else:raise ValueError("metode harus 'bm25' atau 'tfidf'")
    number_lookup=dict(zip(METADATA['doc_id'], METADATA['no_putusan'])) if not METADATA.empty else {}
    items=sorted([(d,v) for d,v in nilai.items() if v>0],key=lambda x:(-x[1],x[0]))[:top_k]
    return pd.DataFrame([{'rank':i,'doc_id':doc,'no_putusan':number_lookup.get(doc,''),'score':round(score,6)}
                          for i,(doc,score) in enumerate(items,1)],columns=['rank','doc_id','no_putusan','score'])

QUERY='putusan pengadilan dan pertimbangan hakim'
HASIL_BM25=cari(QUERY,'bm25',5)
HASIL_TFIDF=cari(QUERY,'tfidf',5)
print('BM25');display(HASIL_BM25)
print('TF-IDF');display(HASIL_TFIDF)
HASIL_BM25.to_csv(OUT_DIR/'hasil_pencarian_bm25.csv',index=False)
HASIL_TFIDF.to_csv(OUT_DIR/'hasil_pencarian_tfidf.csv',index=False)

BM25


,rank,doc_id,no_putusan,score
0,1,Putusan_54_Pid.Sus_2024_PN Tnr,54/Pid.Sus/2024/PN Tnr,5.085124
1,2,Putusan_105_Pid.Sus_2023_PN Tnr,105/Pid.Sus/2023/PN Tnr,0.050040
2,3,Putusan_108_Pid.Sus_2024_PN Tnr,108/Pid.Sus/2024/PN Tnr,0.050040
3,4,Putusan_120_Pid.Sus_2025_PN Tnr,120/Pid.Sus/2025/PN Tnr,0.050040
4,5,Putusan_123_Pid.Sus_2023_PN Tnr,123/Pid.Sus/2023/PN Tnr,0.050040


TF-IDF


,rank,doc_id,no_putusan,score
0,1,Putusan_54_Pid.Sus_2024_PN Tnr,54/Pid.Sus/2024/PN Tnr,0.325369
1,2,Putusan_51_Pid.Sus_2023_PN Tnr,51/Pid.Sus/2023/PN Tnr,0.059467
2,3,Putusan_48_Pid.Sus_2025_PN Tnr,48/Pid.Sus/2025/PN Tnr,0.058836
3,4,Putusan_69_Pid.Sus_2023_PN Tnr,69/Pid.Sus/2023/PN Tnr,0.058647
4,5,Putusan_30_Pid.Sus_2025_PN Tnr,30/Pid.Sus/2025/PN Tnr,0.058374


## 7. Evaluasi optional menggunakan relevansi manual
File `qrels.csv` opsional berisi pasangan `query,doc_id,relevant` yang dinilai manusia. **Tidak menebak label relevansi.** Nilai Precision@5 dihitung hanya jika qrels nyata tersedia.

In [8]:
qrels_path=ROOT/'qrels.csv'
if qrels_path.exists():
    try:
        qrels=pd.read_csv(qrels_path)
        if not {'query','doc_id','relevant'}.issubset(qrels.columns):
            print('qrels.csv harus memiliki kolom query,doc_id,relevant')
        else:
            skor=[]
            for query,frame in qrels.groupby('query'):
                relevant=set(frame.loc[frame['relevant'].astype(str).isin(['1','True','true']),'doc_id'].astype(str))
                pred=cari(query, 'bm25',5)
                top=list(pred['doc_id'])
                p5=sum(doc in relevant for doc in top)/5
                skor.append({'query':query,'precision_at_5':p5,'jumlah_relevan_berlabel':len(relevant)})
            eval_df=pd.DataFrame(skor)
            eval_df.to_csv(OUT_DIR/'evaluasi_qrels.csv',index=False)
            display(eval_df)
    except Exception as e:print('Gagal membaca qrels.csv:',str(e)[:100])
else:
    print('Tidak menghitung Precision@5 tanpa penilaian relevansi manusia; ini bukan error.')

Tidak menghitung Precision@5 tanpa penilaian relevansi manusia; ini bukan error.


## 8. Ringkasan output dan ZIP hasil
Hasil disimpan di `PN_Tanjung_Redeb/hasil_subcpmk4`, sumber tetap di `pdf/` atau `txt/`. File ZIP hasil hanya dibuat jika ada dokumen sungguhan.

In [9]:
if not DOCS:
    print('Belum ada dataset nyata: lakukan pengumpulan PDF, lalu jalankan ulang notebook ini.')
else:
    zpath=ROOT/'Hasil_SUBCPMK4_PN_Tanjung_Redeb.zip'
    with zipfile.ZipFile(zpath,'w',zipfile.ZIP_DEFLATED) as z:
        for f in OUT_DIR.iterdir():
            if f.is_file():z.write(f,'hasil_subcpmk4/'+f.name)
    print('ZIP hasil:',zpath,'| jumlah dokumen:',len(DOCS))
    if Path('/content').exists():print('Download Colab: from google.colab import files; files.download(str(zpath))')

ZIP hasil: C:\Users\marce\KULIAH\Smt7\InformationRetrival\UTS\03_SUBCPMK3_Crawler_dan_Ekstraksi\PN_Tanjung_Redeb\Hasil_SUBCPMK4_PN_Tanjung_Redeb.zip | jumlah dokumen: 50
